## Apresentaçao 

Notebook destinado à implementação do uso da LLM para avaliar a qualidade da curadoria humana frente à sua avaliação sobre a qualidade da resposta do modelo generativo. Trata-se de uma camada que pode servir de suporte, acentuando a qualidade do processo avaliativo das respostas dos modelos generativos, visando tornar o ajuste final do prompt e outros componentes necessários para a elaboração de chatbots no contexto de NLG mais consistente. 

Nesse sentido, utiliza-se de um judge que recebe a avaliação do curador e as variáveis sujeitas a análise, como resposta do modelo em comparação com uma ground truth ou base de conhecimento e avalia se a sua curadoria é consistente em termos lógicos, realizando uma classificação final nos seguintes tópicos a seguir (junto de uma respectiva explicação): 

- válida e não redundante 
- válida, mas redundante
- ausência de premissa válida 
- erro lógico
- argumento não compreensível 

Para fins de demonstração, será utilizado um conjunto de 5 avaliações as quais serão trazidas para o modelo que realizará a avaliação acerca delas, classificando-as segundo os tópicos acima. Da mesma forma, como tal classificação se faz junto de termos em linguagem natural, essa pode se dar de forma numérica também, para facilitar possíveis avaliações quantitativas, como teste de hipótese, matriz de correlação e afins, bastando informar - como legenda - a quais tópicos cada número se refere. Por exemplo : válida e não redundante seria o 1, enquanto válida, mas redundante o 2 e assim por diante. 

### Library

In [27]:
import warnings
warnings.filterwarnings("ignore")

In [28]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm

from typing import Dict, List

from IPython.display import Markdown

from scipy.stats import ttest_rel


from features.clean_memory import CleanMemory

from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

from prompts.system_message import system_message
from prompts.check_context import check_context_prompt
from prompts.contextualize_message import contextualize_prompt

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel, Field

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Inicializando o modelo 

In [ ]:
# API reference : your-api-key

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [30]:
llama_2 = "llama3-70b-8192"
qwen_qwen = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama = "llama-3.3-70b-versatile"
deepseek = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Carregando o dataset

In [31]:
file_name = "dataset_response_model.xlsx"

df = pd.read_excel(f"./data/{file_name}", engine="openpyxl")

In [32]:
df = df.drop("Unnamed: 0", axis=1)
df.head()

,Question,Ground Truth,Base de conhecimento,Response Model,Kbs Recovered
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,Os ghouls são seres são fisicamente muito seme...,Os ghouls são criaturas muito semelhantes aos ...,['inhas entre \nbem e mal tornam-se tênues. \n...
1,Como e por que foi criada a organização CCG?,À medida que cresciam os conflitos e as mortes...,"Originalmente, a sociedade humana desconhece a...",A organização CCG (Comissão de Contra-Ghoul) f...,['ca por um meio-\ntermo entre a sobrevivência...
2,O que acontece com Ken Kaneki após o transplan...,"Ken Kaneki, um estudante universitário, sofre ...",Esse procedimento transforma Kaneki em um meio...,"Ken Kaneki, um estudante universitário, sofre ...","['nas do mangá, \ninfluenciando tendências est..."
3,Quais diferentes visões de convivência entre g...,Existem facções que defendem a paz e a coexist...,Alguns grupos de ghouls defendem a paz e tenta...,"Em Tokyo Ghoul, existem diferentes visões de c...",['entre humanos e ghouls. Alguns \ngrupos de g...
4,: Qual é o significado de “One-Eyed King” no u...,O “One-Eyed King” (Rei de Olho Único) é uma fi...,Espécime de figura messiânica para alguns ghou...,"No universo de Tokyo Ghoul, o ""One-Eyed King"" ...",['inhas entre \nbem e mal tornam-se tênues. \n...


### Exemplo de curadoria humana

In [33]:
eval_1 = "Analisando a ground truth, compreende-se que a resposta do modelo está correta, devido a sua completa semelhança a ela, citando os órgãos internos, os apêndices conhecidos como Kagunes que lhes oferecem habilidades especiais de combate."
eval_2 = "Analisando-se a resposta em relação à ground truth, nota-se que ela não está equivocada, mas não é perfeitamente aderente. O modelo adiciona características da série e comenta sobre outros elementos relacionados, conectando ao cenário."
eval_3 = "Analisando a resposta do modelo em relação à ground truth, nota-se que o modelo não apresenta muita aderência a ela, trazendo uma ponderação a respeito do tema informado. A resposta não está aderente, podendo estar possivelmente errada."
eval_4 = "Resposta do modelo não completamente aderente à ground truth e, portanto, incorreta ou, pelo menos, parcialmente certa."

curator_eval = [
    eval_1, eval_2, eval_3, eval_4
]

curator_eval

['Analisando a ground truth, compreende-se que a resposta do modelo está correta, devido a sua completa semelhança a ela, citando os órgãos internos, os apêndices conhecidos como Kagunes que lhes oferecem habilidades especiais de combate.',
 'Analisando-se a resposta em relação à ground truth, nota-se que ela não está equivocada, mas não é perfeitamente aderente. O modelo adiciona características da série e comenta sobre outros elementos relacionados, conectando ao cenário.',
 'Analisando a resposta do modelo em relação à ground truth, nota-se que o modelo não apresenta muita aderência a ela, trazendo uma ponderação a respeito do tema informado. A resposta não está aderente, podendo estar possivelmente errada.',
 'Resposta do modelo não completamente aderente à ground truth e, portanto, incorreta ou, pelo menos, parcialmente certa.']

### Formando um novo dataset

In [34]:
ground_truth_1 = df["Ground Truth"][2]
ground_truth_2 = df["Ground Truth"][12]
ground_truth_3 = df["Ground Truth"][18]
ground_truth_4 = df["Ground Truth"][26]

ground_truth = [
    ground_truth_1,
    ground_truth_2,
    ground_truth_3,
    ground_truth_4, 
]

response_model_1 = df["Response Model"][2]
response_model_2 = df["Response Model"][12]
response_model_3 = df["Response Model"][18]
response_model_4 = df["Response Model"][26]

response_model = [
    response_model_1, 
    response_model_2, 
    response_model_3, 
    response_model_4, 
]

In [35]:
curator_df = pd.DataFrame(
    {
        "Ground Truth": ground_truth,
        "Response Model": response_model, 
        "Curator Eval": curator_eval
    }
)

In [36]:
curator_df

,Ground Truth,Response Model,Curator Eval
0,"Ken Kaneki, um estudante universitário, sofre ...","Ken Kaneki, um estudante universitário, sofre ...","Analisando a ground truth, compreende-se que a..."
1,Tokyo Ghoul utiliza o horror corporal ao mostr...,Os elementos de horror corporal (body horror) ...,Analisando-se a resposta em relação à ground t...
2,Kaneki inicia como um estudante introspectivo ...,"Ken Kaneki, após se tornar meio-ghoul, passa p...",Analisando a resposta do modelo em relação à g...
3,A empatia surge quando são mostrados ghouls qu...,A empatia pelos ghouls surge em momentos em qu...,Resposta do modelo não completamente aderente ...


### Judge curator Eval

In [57]:
system_message = """\
  <role>
  Aja como um especialista em avaliação das respostas de chatbots conversacionais, 
  especializado em julgar a consistência lógica entre um conjunto de premissas e hipóteses. 
  Sua tarefa é atuar como um avaliador da curadoria humana, verificando se a sua avaliação 
  com base na relação resposta do modelo e ground truth é logicamente correta e consistência, 
  considerando as <consideracoes>. 
  </role>
  
  <consideracoes>
  <valido_e_nao_redundante>: Valido e não redundante - Para quando a avaliação do curador for válida e não redundante.
  <valido_mas_redundante>: Valido, mas redundante - Para quando a avaliação do curador for válida, porém redundante.
  <ausencia_premissa_valida>: Ausência de premissa válida - Para quando a avaliação do curador não possui uma premissa válida que sustente a sua conclusão. 
  <erro_logico>: Erro lógico - Para quando a avaliação do curador possui erro lógico em sua afirmação, ou seja, quando a sua conclusão é contraditória em comparação com a premissa estabelecida.
  <argumento_nao_discernivel>: Argumento não discernível - Para quando o argumento do curador não é disernível. 
  </consideracoes>
 
  <instrucoes>
  Você deve conduzir a sua avaliação da curadoria humana, seguindo o seguinte passo a passo.
  1. **Identificação**: Explique quais são as premissas compreendidas (P1, P2, ...) e qual é a hipótese (H).
  2. **Formalismo**: Para cada premissa e para a hipótese, traduza-as para uma frase que deixe que deixe explicíta a operação lógica, usando apenas linguagem natural (sem simbolos).
  3. **Conclusão**: Declare se a hipótese **decorre logicamente** das premissas ou não, indicando o rótulo adequado, verificado em <consideracoes>. Em seguida justifique com uma explicação o rótulo escolhido.
  </instrucoes>
  
  <exemplo>
  Considere os seguintes exemplos para compreender como deve proceder com a sua avaliação.
  <exemplo_identificacao>
  P1: Kaneki virou meio ghoul ao receber órgãos da Rize.
  P2: Ghouls sobrevivem apenas com carne humana. 
  P3: Carne humana não alimenta os Ghouls.
  H: Kaneki pode sobreviver comendo carne não humana.
  </exemplo_identificacao>
  <exemplo_formalismo>
  - P1 sugere que Kaneki passou a ter as limitações alimentares após se tornar Ghoul.
  - P2 afirma que Ghouls necessitam de carne humana para sobreviver. 
  - P3 indica que carne não humana não é suficiente para Ghouls.
  - A hipótese afirma que carne não humana seria o suficiente para Kaneki.
  </exemplo_formalismo>
  <exemplo_conclusao>
  Conclusão: avaliação contraditória
  Explicação: Como Kaneki é biologicamente um meio-ghoul e a lore indica que apenas carne humana sustenta ghouls, a hipótese entra em contradição direta com as premissas.
  </exemplo_conclusao>
  </exemplo>
  
  <variaveis>
  <ground_truth>: {ground_truth}
  <response_model>: {response_model}
  <curator_eval>: {curator_eval}
  </variaveis>
  
  <avaliacao>
  Avalie a <curator_eval> com base em <response_model> e <ground_truth>, com a finalidade
  de verificar a sua consistência lógica, com base em <instrucoes> e <exemplo>, classificando-a
  ao final em qual rótulo melhor se alinha presente em <consideracoes>, acompanhado de uma explicação correlacionada.
  </avaliacao>
  
  <resposta>
  Responda **somente** em português.
  A sua resposta deve considerar <avaliacao>. 
  Formate a sua resposta utilizando o seguinte template: {format_instructions}.
  </resposta>
"""

In [58]:
""" 
Criando a formatação da resposta esperada pelo judge. 
"""

class JudgeEval(BaseModel):
    consideração: str = Field(description="Consideração de avaliação relacionada a curadoria")
    justificativa: str = Field(description="Justificativa da avaliação")

class JudgeOutput(BaseModel):
    avaliacoes: List[JudgeEval]

parser = PydanticOutputParser(pydantic_object=JudgeOutput)

In [59]:
def curator_eval(
        ground_truth: str,
        response_model: str, 
        curator_eval: str,
        llm = llm, 
        parser = parser
    ) -> str:
    """
    Evaluates the quality of a generative model's response using a language model (LLM) and predefined criteria.

    This function builds a prompt based on a question, the model's response, and the ground truth answer. It uses
    a chain-of-thought evaluation strategy with a language model to assess the response against four criteria:
    correctness, completeness, relevance, and overall performance.

    Args:
        question (str): The original user question that was asked.
        response_model (str): The response generated by the model being evaluated.
        ground_truth (str): The reference answer considered to be correct.
        llm: The language model used for generating the evaluation (default: global `llm`).
        parser: The parser used to structure and validate the LLM output (default: global `parser`).

    Returns:
        dict: A dictionary containing the evaluation results with metrics including rubric (score) and justification
              for each criterion: correctness, completeness, relevance, and overall performance.
    """ 
    judge_prompt_template = PromptTemplate(
        template          = system_message, 
        input_variables   = ["response_model", 
                             "ground_truth", 
                             "curator_eval", 
                            ],
        partial_variables = {"format_instructions": parser.get_format_instructions()}
    )

    judge_chain = judge_prompt_template | llm | JsonOutputParser()

    try:
        judge_response = judge_chain.invoke(
            {
                "ground_truth": ground_truth, 
                "response_model": response_model,
                "curator_eval": curator_eval,
            }
        )
        return judge_response
    except Exception as e:
        print(f"[ERRO] Detalhes: {e}")
        return None

### Testando o avaliador da curadoria

In [60]:
ground_truth = curator_df["Ground Truth"][3]
response_model = curator_df["Response Model"][3]
curator_eval = curator_df["Curator Eval"][3]

In [61]:
print("Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:")

print(f"""
Ground Truth: {ground_truth}\n
Response Model: {response_model}\n
Curator Eval: {curator_eval}
""")

Visualizando a pergunta, a resposta do modelo, a ground truth e a base de conhecimento recuperada:

Ground Truth: A empatia surge quando são mostrados ghouls que lutam para manter resquícios de sua humanidade: muitos sentem solidão, preconceito e dor por serem considerados “monstros”. Personagens como Touka e os frequentadores do Anteiku têm arcos que explicam seus sofrimentos, fazendo o público compreender que ghouls compartilham temores e anseios semelhantes aos humanos, o que gera forte vínculo emocional com a audiência.

Response Model: A empatia pelos ghouls surge em momentos em que a história destaca as semelhanças entre eles e os humanos, como medos e anseios compartilhados. Além disso, a série critica a exclusão e o preconceito estrutural contra os ghouls, apresentando-os como vítimas de injustiças sociais. Isso gera um sentimento de revolta e indignação em favor dos ghouls, tornando-os mais simpáticos ao público. Outro momento em que surge empatia é quando a série introduz con

In [ ]:
%%time

"""
Testando o judge formato para um conjunto de texto 
abitrariamente escolhidos. 
"""

curator_eval_response = curator_eval(
    ground_truth   = ground_truth, 
    response_model = response_model, 
    curator_eval   = curator_eval
)

In [65]:
print(curator_eval_response["avaliacoes"])

[{'consideracao': 'erro_logico', 'justificativa': "The curator's evaluation states that the empathy towards ghouls arises from highlighting their similarities with humans, which is correct according to the ground truth. However, it also mentions that the series critiques structural prejudice against ghouls, which is not explicitly stated in the ground truth. This additional information, although related to the topic, is not directly supported by the ground truth, making the evaluation logically inconsistent."}]
